# LeetCode 1023 — Camelcase Matching bằng Python

Link đề: https://leetcode.com/problems/camelcase-matching/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Two Pointers (hai con trỏ), String (chuỗi), Trie, String Matching (khớp chuỗi)

Skeleton LeetCode:

```python
class Solution:
    def camelMatch(self, queries: List[str], pattern: str) -> List[bool]:
        ...
```

Ghi chú: cho danh sách `queries` và một `pattern`. Một từ truy vấn **khớp** với `pattern` nếu có thể **chèn các chữ cái thường** (chỉ chữ thường!) vào `pattern` để thu được đúng từ truy vấn (được chèn ở vị trí bất kỳ, hoặc không chèn gì). Nói cách khác: `pattern` phải là **chuỗi con không liên tiếp** (subsequence) của truy vấn, **và** mọi ký tự còn lại (không thuộc phần khớp) đều phải là **chữ thường**. Cách giải gọn là **hai con trỏ**: duyệt truy vấn, khớp từng ký tự của `pattern` theo thứ tự; ký tự không khớp mà là **chữ hoa** thì loại ngay. Đây là bài "khớp chuỗi" kiểu CamelCase (như tìm kiếm tên lớp `FB` -> `FooBar`).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: khớp = `pattern` là chuỗi con không liên tiếp của truy vấn, mọi ký tự **thừa** là chữ thường.
2. Nắm các công cụ Python cần thiết: `str.isupper()`/`str.islower()`, hai con trỏ trên hai chuỗi, `re.fullmatch` với lớp ký tự `[a-z]*`, `lru_cache`, và `itertools.combinations` để vét cạn tập vị trí.
3. Hiểu nguyên lý **tham lam hai con trỏ**: khớp ký tự `pattern` với ký tự truy vấn sớm nhất có thể; ký tự truy vấn không khớp mà là **chữ hoa** làm khớp thất bại; cuối cùng cần đã **dùng hết** `pattern`.
4. Đi qua 4 cách: thử mọi tập vị trí bị "chèn" trong truy vấn (brute force), quy hoạch động / đệ quy có nhớ theo `(i, j)` (xét cả hai lựa chọn khớp hoặc bỏ qua), hai con trỏ tham lam (cách nộp chính) và biểu thức chính quy (`re`).
5. Tránh các lỗi kinh điển: chỉ kiểm tra chuỗi con không liên tiếp mà quên điều kiện "ký tự thừa phải là chữ thường", dừng sớm khi đã khớp hết `pattern` mà bỏ qua các chữ hoa còn lại, quên kiểm tra đã dùng hết `pattern`, so sánh không phân biệt hoa/thường, và chỉ so các chữ hoa mà bỏ qua chữ thường của `pattern`.

## 2. Đề bài & ví dụ

Cho mảng chuỗi `queries` và chuỗi `pattern`, trả về mảng boolean `answer` với `answer[i]` là `true` nếu `queries[i]` **khớp** với `pattern`, ngược lại `false`.

Từ truy vấn `queries[i]` **khớp** `pattern` nếu ta có thể **chèn các chữ cái tiếng Anh viết thường** vào `pattern` để được đúng `queries[i]`. Có thể chèn ở **vị trí bất kỳ**, hoặc **không chèn** gì cả.

**Ví dụ 1:** `queries = ["FooBar","FooBarTest","FootBall","FrameBuffer","ForceFeedBack"]`, `pattern = "FB"` -> `[true,false,true,true,false]`.

- `"FooBar"` = `"F"` + `"oo"` + `"B"` + `"ar"`.
- `"FootBall"` = `"F"` + `"oot"` + `"B"` + `"all"`.
- `"FrameBuffer"` = `"F"` + `"rame"` + `"B"` + `"uffer"`.

**Ví dụ 2:** cùng `queries`, `pattern = "FoBa"` -> `[true,false,true,false,false]`.

**Ví dụ 3:** cùng `queries`, `pattern = "FoBaT"` -> `[false,true,false,false,false]` (`"FooBarTest"` = `"Fo"` + `"o"` + `"Ba"` + `"r"` + `"T"` + `"est"`).

**Ràng buộc:**

- `1 <= pattern.length, queries.length <= 100`
- `1 <= queries[i].length <= 100`
- `queries[i]` và `pattern` gồm các chữ cái tiếng Anh (hoa và thường).

In [1]:
from typing import List, Optional
from functools import lru_cache
from itertools import combinations
import re
import random, time

QUERIES = ["FooBar", "FooBarTest", "FootBall", "FrameBuffer", "ForceFeedBack"]
examples = [
    (QUERIES, "FB", [True, False, True, True, False]),
    (QUERIES, "FoBa", [True, False, True, False, False]),
    (QUERIES, "FoBaT", [False, True, False, False, False]),
]
for queries, pattern, expected in examples:
    print(f"pattern={pattern!r} -> kỳ vọng {expected}")

pattern='FB' -> kỳ vọng [True, False, True, True, False]
pattern='FoBa' -> kỳ vọng [True, False, True, False, False]
pattern='FoBaT' -> kỳ vọng [False, True, False, False, False]


## 3. Kiến thức nền

### 3.1. Chữ hoa, chữ thường và chuỗi con không liên tiếp

`c.isupper()` kiểm tra `c` là chữ hoa, `c.islower()` kiểm tra chữ thường. `pattern` là **chuỗi con không liên tiếp** (subsequence) của `query` nếu xóa vài ký tự của `query` (giữ nguyên thứ tự) thu được `pattern`. Bài này thêm ràng buộc: các ký tự bị xóa (ký tự "thừa") phải là **chữ thường**.

In [2]:
print("F".isupper(), "o".islower(), "FooBar".isupper(), "foobar".islower())
def is_subsequence(p, q):
    it = iter(q)
    return all(ch in it for ch in p)
print("FB là chuỗi con không liên tiếp của FooBar:", is_subsequence("FB", "FooBar"), "| của FooBarTest:", is_subsequence("FB", "FooBarTest"))

True True False True
FB là chuỗi con không liên tiếp của FooBar: True | của FooBarTest: True


### 3.2. Hai con trỏ

Giữ con trỏ `j` trên `pattern` (ký tự cần khớp tiếp theo). Duyệt từng ký tự `c` của truy vấn:

- nếu `j < len(pattern)` và `c == pattern[j]`: khớp, `j += 1`;
- ngược lại nếu `c` là **chữ hoa**: ký tự thừa là chữ hoa ⇒ **không khớp**;
- ngược lại (`c` chữ thường thừa): bỏ qua.

Cuối cùng khớp ⇔ `j == len(pattern)`.

In [3]:
def trace_pointer(query, pattern):
    j = 0
    for c in query:
        if j < len(pattern) and c == pattern[j]:
            print(f"{c!r}: khớp pattern[{j}] = {pattern[j]!r} -> j = {j + 1}")
            j += 1
        elif c.isupper():
            print(f"{c!r}: chữ hoa thừa -> KHÔNG khớp")
            return False
        else:
            print(f"{c!r}: chữ thường thừa -> bỏ qua")
    print("đã dùng hết pattern?", j == len(pattern))
    return j == len(pattern)

assert trace_pointer("FooBar", "FB") is True
assert trace_pointer("FooBarTest", "FB") is False

'F': khớp pattern[0] = 'F' -> j = 1
'o': chữ thường thừa -> bỏ qua
'o': chữ thường thừa -> bỏ qua
'B': khớp pattern[1] = 'B' -> j = 2
'a': chữ thường thừa -> bỏ qua
'r': chữ thường thừa -> bỏ qua
đã dùng hết pattern? True
'F': khớp pattern[0] = 'F' -> j = 1
'o': chữ thường thừa -> bỏ qua
'o': chữ thường thừa -> bỏ qua
'B': khớp pattern[1] = 'B' -> j = 2
'a': chữ thường thừa -> bỏ qua
'r': chữ thường thừa -> bỏ qua
'T': chữ hoa thừa -> KHÔNG khớp


### 3.3. Biểu thức chính quy

Mẫu `^[a-z]*F[a-z]*B[a-z]*$` khớp mọi chuỗi có dạng "chữ thường tùy ý, `F`, chữ thường tùy ý, `B`, chữ thường tùy ý". Chèn `[a-z]*` **trước, giữa và sau** các ký tự của `pattern` đúng bằng quy tắc của đề. Pattern chỉ gồm chữ cái nên không cần thoát ký tự đặc biệt (dù `re.escape` vẫn an toàn).

In [4]:
rx = re.compile("[a-z]*" + "[a-z]*".join(re.escape(c) for c in "FB") + "[a-z]*")
print(rx.pattern)
print([bool(rx.fullmatch(q)) for q in QUERIES])

[a-z]*F[a-z]*B[a-z]*
[True, False, True, True, False]


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `pattern = "FB"`.

- `"FooBar"`: `F` khớp `F`; `o`, `o` chữ thường thừa; `B` khớp `B`; `a`, `r` thừa thường ⇒ `true`.
- `"FooBarTest"`: như trên cho tới hết `FB`, rồi `T` là chữ **hoa** thừa ⇒ `false` (chỉ chèn được chữ thường).
- `"FootBall"`: `F`, `o`,`o`,`t` thừa, `B`, `a`,`l`,`l` thừa ⇒ `true`.
- `"FrameBuffer"`: `F`, `rame`, `B`, `uffer` ⇒ `true`.
- `"ForceFeedBack"`: `F` khớp, `orce` thừa, `F` (chữ hoa thứ hai) không khớp `B` và là chữ hoa ⇒ `false`.

Ví dụ 3: `pattern = "FoBaT"` với `"FooBarTest"`: `F`, `o` khớp `Fo`; `o` thừa; `B`, `a` khớp `Ba`; `r` thừa; `T` khớp `T`; `est` thừa ⇒ `true`.

## 5. Cách 1 — Brute Force: thử mọi tập vị trí "chèn"

**Ý tưởng.** Làm đúng **theo định nghĩa**: truy vấn `q` khớp nếu tồn tại tập vị trí `S` trong `q` (các ký tự được "chèn") sao cho (a) mọi ký tự ở `S` là chữ thường, và (b) xóa các vị trí `S` khỏi `q` được đúng `pattern`. Thử mọi cách chọn `S` có kích thước `len(q) - len(pattern)` bằng `itertools.combinations`. Không dùng tham lam hay quy hoạch động nên đây là **oracle độc lập** (chỉ cho truy vấn ngắn).

In [5]:
def match_brute(q: str, pattern: str) -> bool:
    k = len(q) - len(pattern)
    if k < 0:
        return False
    for inserted in combinations(range(len(q)), k):
        ins = set(inserted)
        if all(q[i].islower() for i in ins) and "".join(q[i] for i in range(len(q)) if i not in ins) == pattern:
            return True
    return False

def camel_brute(queries: List[str], pattern: str) -> List[bool]:
    return [match_brute(q, pattern) for q in queries]

for queries, pattern, expected in examples:
    got = camel_brute(queries, pattern)
    print(f"pattern={pattern!r}: brute = {got}", "OK" if got == expected else "SAI")
    assert got == expected

pattern='FB': brute = [True, False, True, True, False] OK
pattern='FoBa': brute = [True, False, True, False, False] OK
pattern='FoBaT': brute = [False, True, False, False, False] OK


**Độ phức tạp và nút thắt.** `C(|q|, |q| - |p|)` cách chọn, mỗi cách `O(|q|)`: tăng rất nhanh theo độ dài (với `|q| = 100` và `|p| = 10`: `C(100, 90) ~ 1.7 * 10^13`). Đo thử:

In [6]:
import math
for L in (10, 14, 18, 20):
    q = "Ab" + "c" * (L - 3) + "D"
    t0 = time.perf_counter()
    match_brute(q, "AbD" if L > 3 else "A")
    print(f"|q|={L:2}, |p|=3: C({L}, {L - 3}) = {math.comb(L, L - 3):>9}, {(time.perf_counter() - t0) * 1000:8.2f} ms")
print("|q| = 100, |p| = 10: C(100, 90) ~ 1.7 * 10^13 -> không thể")

|q|=10, |p|=3: C(10, 7) =       120,     0.17 ms
|q|=14, |p|=3: C(14, 11) =       364,     0.20 ms
|q|=18, |p|=3: C(18, 15) =       816,     0.49 ms
|q|=20, |p|=3: C(20, 17) =      1140,     0.76 ms
|q| = 100, |p| = 10: C(100, 90) ~ 1.7 * 10^13 -> không thể


Nút thắt: ta thử mọi tập vị trí, trong khi chỉ cần **một** cách ghép hợp lệ. Quy hoạch động theo `(i, j)` kiểm cả hai lựa chọn ở từng bước mà không lặp lại việc.

## 6. Ý tưởng thuật toán / bước đột phá

**Trạng thái `(i, j)`: đã xét `i` ký tự đầu của truy vấn và khớp `j` ký tự đầu của `pattern`.**

Tại ký tự `q[i]` có tối đa hai lựa chọn:

- **khớp** với `pattern[j]` (nếu `q[i] == pattern[j]`): sang `(i + 1, j + 1)`;
- **coi là ký tự chèn** (nếu `q[i]` là chữ **thường**): sang `(i + 1, j)`.

Chữ **hoa** không được chèn nên chỉ còn lựa chọn khớp. Đích: `i = len(q)` và `j = len(pattern)`.

Quan sát thêm: lựa chọn **khớp sớm nhất** luôn an toàn (tham lam): nếu `q[i] == pattern[j]` thì khớp ngay cũng không làm xấu đi (xem mục 10). Khi đó mỗi `(i, j)` chỉ còn một lựa chọn: hai con trỏ `O(|q|)`.

| Cách | Ý tưởng |
|---|---|
| 2 | Đệ quy có nhớ trên `(i, j)`, thử cả hai lựa chọn |
| 3 | Hai con trỏ tham lam (nộp) |
| 4 | Biểu thức chính quy `[a-z]*P0[a-z]*P1...[a-z]*` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Quy hoạch động / đệ quy có nhớ

**Giải thích từng đoạn.**

- `f(i, j)` = truy vấn từ ký tự `i` có thể khớp phần còn lại của `pattern` bắt đầu ở `j` không.
- Nếu `i == len(q)`: đúng khi `j == len(pattern)`.
- Nếu `j < len(pattern)` và `q[i] == pattern[j]`: thử khớp `f(i + 1, j + 1)`.
- Nếu `q[i]` là chữ thường: thử coi là chèn `f(i + 1, j)`.
- Hai lựa chọn hợp bằng `or`. Độ sâu đệ quy tới `|q| <= 100`.

In [7]:
def match_dp(q: str, pattern: str) -> bool:
    n, m = len(q), len(pattern)

    @lru_cache(None)
    def f(i: int, j: int) -> bool:
        if i == n:
            return j == m
        ok = False
        if j < m and q[i] == pattern[j]:
            ok = f(i + 1, j + 1)                       # khớp
        if not ok and q[i].islower():
            ok = f(i + 1, j)                           # chữ thường: coi là ký tự chèn
        return ok

    return f(0, 0)

def camel_dp(queries: List[str], pattern: str) -> List[bool]:
    return [match_dp(q, pattern) for q in queries]

for queries, pattern, expected in examples:
    got = camel_dp(queries, pattern)
    print(f"pattern={pattern!r}: DP = {got}", "OK" if got == expected else "SAI")
    assert got == expected

pattern='FB': DP = [True, False, True, True, False] OK
pattern='FoBa': DP = [True, False, True, False, False] OK
pattern='FoBaT': DP = [False, True, False, False, False] OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def camelMatch(self, queries: List[str], pattern: str) -> List[bool]:
        from functools import lru_cache
        m = len(pattern)

        def match(q: str) -> bool:
            n = len(q)

            @lru_cache(None)
            def f(i: int, j: int) -> bool:
                if i == n:
                    return j == m
                if j < m and q[i] == pattern[j] and f(i + 1, j + 1):   # khớp ký tự pattern
                    return True
                return q[i].islower() and f(i + 1, j)                   # chữ thường: coi là ký tự chèn

            return f(0, 0)

        return [match(q) for q in queries]

sol_dp = Solution().camelMatch
for queries, pattern, expected in examples:
    assert sol_dp(queries, pattern) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** mỗi truy vấn `O(|q| * |p|)` trạng thái, mỗi trạng thái `O(1)`; bộ nhớ `O(|q| * |p|)`; tổng `O(Q * |q| * |p|)` (`<= 100 * 100 * 100 = 10^6`).

---

### 7.2. Cách 3 — Hai con trỏ tham lam (cách nộp chính)

**Giải thích từng đoạn.**

- `j` là con trỏ trên `pattern` (ký tự cần khớp kế tiếp), ban đầu `0`.
- Với mỗi ký tự `c` của truy vấn: nếu `j < len(pattern)` và `c == pattern[j]` thì `j += 1` (khớp sớm nhất có thể).
- Ngược lại nếu `c` là **chữ hoa**: trả `False` (chữ hoa thừa không được chèn, không khớp được).
- Ngược lại (chữ thường thừa): bỏ qua.
- Kết quả: `j == len(pattern)` (đã dùng hết `pattern`).

In [9]:
def match_pointers(q: str, pattern: str) -> bool:
    j, m = 0, len(pattern)
    for c in q:
        if j < m and c == pattern[j]:
            j += 1
        elif c.isupper():
            return False
    return j == m

def camel_pointers(queries: List[str], pattern: str) -> List[bool]:
    return [match_pointers(q, pattern) for q in queries]

for queries, pattern, expected in examples:
    got = camel_pointers(queries, pattern)
    print(f"pattern={pattern!r}: hai con trỏ = {got}", "OK" if got == expected else "SAI")
    assert got == expected

pattern='FB': hai con trỏ = [True, False, True, True, False] OK
pattern='FoBa': hai con trỏ = [True, False, True, False, False] OK
pattern='FoBaT': hai con trỏ = [False, True, False, False, False] OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def camelMatch(self, queries: List[str], pattern: str) -> List[bool]:
        def match(q: str) -> bool:
            j, m = 0, len(pattern)
            for c in q:
                if j < m and c == pattern[j]:
                    j += 1                      # khớp ký tự pattern sớm nhất có thể
                elif c.isupper():
                    return False                # chữ hoa thừa: không thể là ký tự chèn
            return j == m                       # phải dùng hết pattern

        return [match(q) for q in queries]

sol_pointers = Solution().camelMatch
for queries, pattern, expected in examples:
    assert sol_pointers(queries, pattern) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(Q * |q|)` (mỗi truy vấn một lượt), bộ nhớ `O(1)` thêm.

---

### 7.3. Cách 4 — Biểu thức chính quy

**Ý tưởng khác.** Dựng một biểu thức chính quy từ `pattern`: `[a-z]*` + `p0` + `[a-z]*` + `p1` + ... + `p_{k-1}` + `[a-z]*`, rồi dùng `re.fullmatch` cho từng truy vấn. `[a-z]*` đại diện cho phần **chèn chữ thường** (có thể rỗng) ở trước, giữa và sau các ký tự của `pattern`. `re.escape` để an toàn nếu `pattern` chứa ký tự đặc biệt (ở đây chỉ có chữ cái).

**Giải thích từng đoạn.** Biên dịch một lần (`re.compile`) rồi dùng cho mọi truy vấn.

In [11]:
def camel_regex(queries: List[str], pattern: str) -> List[bool]:
    rx = re.compile("[a-z]*" + "[a-z]*".join(re.escape(c) for c in pattern) + "[a-z]*")
    return [rx.fullmatch(q) is not None for q in queries]

for queries, pattern, expected in examples:
    got = camel_regex(queries, pattern)
    print(f"pattern={pattern!r}: regex = {got}", "OK" if got == expected else "SAI")
    assert got == expected

pattern='FB': regex = [True, False, True, True, False] OK
pattern='FoBa': regex = [True, False, True, False, False] OK
pattern='FoBaT': regex = [False, True, False, False, False] OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def camelMatch(self, queries: List[str], pattern: str) -> List[bool]:
        import re
        # [a-z]* đại diện cho phần chèn chữ thường (có thể rỗng) trước, giữa và sau các ký tự của pattern
        rx = re.compile("[a-z]*" + "[a-z]*".join(re.escape(c) for c in pattern) + "[a-z]*")
        return [rx.fullmatch(q) is not None for q in queries]

sol_regex = Solution().camelMatch
for queries, pattern, expected in examples:
    assert sol_regex(queries, pattern) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** phụ thuộc bộ máy `re`. Các nhóm `[a-z]*` lồng nhau xen các ký tự cố định có thể **quay lui theo hàm mũ** khi truy vấn **không khớp** và `pattern` có nhiều chữ thường (xem số đo ở mục 11), nên cách này gọn nhưng **không an toàn** cho truy vấn dài; nên ưu tiên hai con trỏ. Bộ nhớ `O(|p|)`.

## 8. TRACE từng bước

### 8.1. Hai con trỏ cho `"FooBarTest"` với `pattern = "FoBaT"`

In từng ký tự.

In [13]:
assert trace_pointer("FooBarTest", "FoBaT") is True
print("---")
assert trace_pointer("ForceFeedBack", "FB") is False

'F': khớp pattern[0] = 'F' -> j = 1
'o': khớp pattern[1] = 'o' -> j = 2
'o': chữ thường thừa -> bỏ qua
'B': khớp pattern[2] = 'B' -> j = 3
'a': khớp pattern[3] = 'a' -> j = 4
'r': chữ thường thừa -> bỏ qua
'T': khớp pattern[4] = 'T' -> j = 5
'e': chữ thường thừa -> bỏ qua
's': chữ thường thừa -> bỏ qua
't': chữ thường thừa -> bỏ qua
đã dùng hết pattern? True
---
'F': khớp pattern[0] = 'F' -> j = 1
'o': chữ thường thừa -> bỏ qua
'r': chữ thường thừa -> bỏ qua
'c': chữ thường thừa -> bỏ qua
'e': chữ thường thừa -> bỏ qua
'F': chữ hoa thừa -> KHÔNG khớp


### 8.2. Quy hoạch động: các trạng thái `(i, j)` hợp lệ cho `"FooBar"`, `pattern = "FB"`

In bảng `f(i, j)` với `i = 0..6`, `j = 0..2`.

In [14]:
def dp_table(q, pattern):
    n, m = len(q), len(pattern)
    f = [[False] * (m + 1) for _ in range(n + 1)]
    f[n][m] = True
    for i in range(n - 1, -1, -1):
        for j in range(m + 1):
            ok = False
            if j < m and q[i] == pattern[j] and f[i + 1][j + 1]:
                ok = True
            if not ok and q[i].islower() and f[i + 1][j]:
                ok = True
            f[i][j] = ok
    return f

f = dp_table("FooBar", "FB")
print("i\\j   0      1      2")
for i, row in enumerate(f):
    print(f"{i} {('' if i == 6 else 'FooBar'[i]):>1}  " + "  ".join(f"{str(x):5}" for x in row))
assert f[0][0] is True

i\j   0      1      2
0 F  True   False  False
1 o  False  True   False
2 o  False  True   False
3 B  False  True   False
4 a  False  False  True 
5 r  False  False  True 
6    False  False  True 


### 8.3. Vì sao quên điều kiện "chữ thường" là sai

`"FooBarTest"` chứa `"FB"` như chuỗi con không liên tiếp, nhưng ký tự `T` là chữ hoa thừa nên **không** khớp.

In [15]:
print("FB là chuỗi con không liên tiếp của FooBarTest:", is_subsequence("FB", "FooBarTest"), "| nhưng khớp theo đề:", match_pointers("FooBarTest", "FB"))
assert is_subsequence("FB", "FooBarTest") and not match_pointers("FooBarTest", "FB")

FB là chuỗi con không liên tiếp của FooBarTest: True | nhưng khớp theo đề: False


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (hai con trỏ tham lam): `O(Q * |q|)`, ngắn gọn, không cần thư viện.

```python
class Solution:
    def camelMatch(self, queries: List[str], pattern: str) -> List[bool]:
        def match(q: str) -> bool:
            j, m = 0, len(pattern)
            for c in q:
                if j < m and c == pattern[j]:
                    j += 1                      # khớp ký tự pattern sớm nhất có thể
                elif c.isupper():
                    return False                # chữ hoa thừa: không thể là ký tự chèn
            return j == m                       # phải dùng hết pattern

        return [match(q) for q in queries]
```

In [16]:
class Solution:
    def camelMatch(self, queries: List[str], pattern: str) -> List[bool]:
        def match(q: str) -> bool:
            j, m = 0, len(pattern)
            for c in q:
                if j < m and c == pattern[j]:
                    j += 1                      # khớp ký tự pattern sớm nhất có thể
                elif c.isupper():
                    return False                # chữ hoa thừa: không thể là ký tự chèn
            return j == m                       # phải dùng hết pattern

        return [match(q) for q in queries]

sol = Solution()
for queries, pattern, expected in examples:
    got = sol.camelMatch(queries, pattern)
    print(f"pattern={pattern!r}: {got}", "OK" if got == expected else "SAI")
    assert got == expected

pattern='FB': [True, False, True, True, False] OK
pattern='FoBa': [True, False, True, False, False] OK
pattern='FoBaT': [False, True, False, False, False] OK


## 10. Vì sao đúng

**Đặc trưng.** `q` khớp `pattern` ⇔ tồn tại các chỉ số `i_0 < i_1 < ... < i_{m-1}` với `q[i_k] = pattern[k]` và mọi ký tự `q[i]` với `i` không thuộc `{i_k}` là chữ thường. (Đúng định nghĩa: các ký tự không thuộc dãy `i_k` chính là những ký tự được chèn.)

**Tham lam đúng (phân đoạn).** Mọi chữ hoa của `q` đều phải được ghép (không được coi là ký tự chèn), mà chữ thường của `pattern` không ghép được với chữ hoa của `q`. Vì vậy ta viết `q = s_0 U_1 s_1 U_2 ... U_r s_r` (các `U_i` là chữ hoa, các `s_i` là chuỗi chữ thường, có thể rỗng) và `pattern = p_0 V_1 p_1 ... V_t p_t` tương tự. Khớp ⇔ `r = t`, `U_i = V_i` với mọi `i`, và **mỗi `p_i` là chuỗi con không liên tiếp của `s_i`** (các chữ thường của `p_i` chỉ có thể ghép với chữ thường của `s_i` cùng đoạn, theo thứ tự). Hai con trỏ làm đúng việc đó đoạn này sau đoạn khác:

- trong đoạn `s_i` nó ghép `p_i` theo cách tham lam sớm nhất, vốn là cách chuẩn kiểm tra chuỗi con không liên tiếp (đúng nếu và chỉ nếu `p_i` là chuỗi con không liên tiếp của `s_i`);
- gặp chữ hoa `U_{i+1}` của `q`: nếu `p_i` chưa được ghép hết thì `pattern[j]` là chữ thường khác `U_{i+1}` và hàm trả `False` (đúng, vì `p_i` không là chuỗi con của `s_i`); nếu `p_i` đã hết thì cần `pattern[j] = V_{i+1} = U_{i+1}`, ngược lại trả `False` (đúng: chữ hoa không khớp);
- cuối cùng `j == m` ⇔ đã ghép hết đoạn `p_t` (và mọi chữ hoa của `pattern` đều được khớp).

Chiều ngược lại hiển nhiên: nếu hai con trỏ trả `True` thì các vị trí đã khớp là một cách ghép hợp lệ (mọi ký tự không được khớp đều là chữ thường vì chữ hoa không khớp đã trả `False`).

Ta còn **kiểm chứng bằng máy** (bên dưới): hai con trỏ khớp oracle theo định nghĩa trên **mọi** cặp `(q, pattern)` với ký tự trong `{a, b, A, B}`, độ dài `q <= 5`, `pattern <= 3`.

**Cách 2 (DP)** xét **cả hai** lựa chọn nên đúng theo định nghĩa và là cơ sở đối chiếu độc lập với tham lam. **Cách 4 (regex)** biểu diễn đúng ngôn ngữ "chèn chữ thường vào `pattern`".

**Kiểm chứng bằng máy.** (1) Oracle theo định nghĩa (Cách 1, thử mọi tập vị trí chèn) so với DP, hai con trỏ, regex trên mọi cặp `(q, pattern)` kể trên; (2) DP là oracle cho truy vấn dài (tới 100) trên nhiều ca ngẫu nhiên có seed.

In [17]:
from itertools import product
alphabet = "abAB"
total = matched = 0
for lq in range(1, 6):
    for qt in product(alphabet, repeat=lq):
        q = "".join(qt)
        for lp in range(1, 4):
            for pt in product(alphabet, repeat=lp):
                p = "".join(pt)
                ref = match_brute(q, p)
                assert match_dp(q, p) == match_pointers(q, p) == camel_regex([q], p)[0] == ref, (q, p)
                total += 1
                matched += ref
print(f"đã đối chiếu {total} cặp (q, pattern): {matched} khớp, {total - matched} không khớp; các cách khớp oracle")

đã đối chiếu 114576 cặp (q, pattern): 3784 khớp, 110792 không khớp; các cách khớp oracle


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi tập vị trí chèn | `O(C(\|q\|, \|q\| - \|p\|) * \|q\|)` mỗi truy vấn | `O(\|q\|)` |
| 2. DP `(i, j)` | `O(\|q\| * \|p\|)` mỗi truy vấn | `O(\|q\| * \|p\|)` |
| 3. Hai con trỏ | `O(\|q\|)` mỗi truy vấn | `O(1)` |
| 4. Regex | `O(\|q\| * \|p\|)` tồi nhất | `O(\|p\|)` |

Đo ở cỡ tối đa: `100` truy vấn độ dài `100`, `pattern` độ dài `100` (mọi ký tự khớp) và các ca khác. (Cách 4 bằng `re` **không** đo ở cỡ này vì có thể quay lui rất nhiều, xem ghi chú ở mục 7.3 và đo riêng ở ca ngắn bên dưới.)

In [18]:
rng = random.Random(11)
def rand_word(L, p_upper=0.2):
    return "".join(rng.choice("ABCDEFGH") if rng.random() < p_upper else rng.choice("abcdefgh") for _ in range(L))

pattern = "A" + "b" * 98 + "C"
queries_hard = ["A" + "b" * 98 + "C" for _ in range(100)] + ["A" + "b" * 97 + "c" + "C" for _ in range(0)]
queries_rand = [rand_word(100) for _ in range(100)]
queries_long = ["a" * 99 + "A" for _ in range(100)]
for label, qs, pat in (("khớp hoàn toàn", queries_hard, pattern), ("ngẫu nhiên", queries_rand, "ab"), ("chữ thường dài", queries_long, "aaaA")):
    ref = camel_pointers(qs, pat)
    row = [f"{label:16}"]
    for name, fn in (("DP", sol_dp), ("hai con trỏ", sol_pointers), ("nộp", sol.camelMatch)):
        t0 = time.perf_counter()
        got = fn(qs, pat)
        dt = (time.perf_counter() - t0) * 1000
        assert got == ref, (label, name)
        row.append(f"{name} {dt:7.2f} ms")
    print(" | ".join(row), "| số khớp:", sum(ref))

# regex: đo ở truy vấn ngắn, và minh họa quay lui khi KHÔNG khớp
short_q = ["FooBar", "FooBarTest", "FootBall", "FrameBuffer", "ForceFeedBack"] * 20
t0 = time.perf_counter(); sol_regex(short_q, "FB"); print(f"regex (100 truy vấn ngắn): {(time.perf_counter() - t0) * 1000:.2f} ms")
for L in (12, 16, 20, 22):
    q, pat = "a" * L + "A", "a" * (L // 2 + 1) + "B"          # không khớp: regex thử rất nhiều cách chia các chữ thường
    t0 = time.perf_counter(); r1 = sol_regex([q], pat); dt1 = time.perf_counter() - t0
    t0 = time.perf_counter(); r2 = sol_pointers([q], pat); dt2 = time.perf_counter() - t0
    assert r1 == r2 == [False]
    print(f"|q|={L + 1:2}: regex {dt1 * 1000:9.2f} ms | hai con trỏ {dt2 * 1000:7.4f} ms")

khớp hoàn toàn   | DP    3.23 ms | hai con trỏ    0.35 ms | nộp    0.33 ms | số khớp: 100
ngẫu nhiên       | DP    0.37 ms | hai con trỏ    0.04 ms | nộp    0.05 ms | số khớp: 0
chữ thường dài   | DP    3.68 ms | hai con trỏ    0.83 ms | nộp    0.52 ms | số khớp: 100
regex (100 truy vấn ngắn): 0.09 ms
|q|=13: regex      0.23 ms | hai con trỏ  0.0034 ms
|q|=17: regex      1.21 ms | hai con trỏ  0.0034 ms
|q|=21: regex     16.18 ms | hai con trỏ  0.0074 ms
|q|=23: regex     63.99 ms | hai con trỏ  0.0077 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), đối chiếu mọi cách, test ngẫu nhiên có seed (đúng ràng buộc) và các tính chất.

In [19]:
def all_solutions():
    return {"dp": camel_dp, "pointers": camel_pointers, "sol_dp": sol_dp, "sol_pointers": sol_pointers, "final": sol.camelMatch}

def check_all(queries, pattern, expected=None, brute=True):
    results = {name: fn(list(queries), pattern) for name, fn in all_solutions().items()}
    ref = results["pointers"]
    assert all(r == ref for r in results.values()), (queries[:3], pattern, results)
    # regex chỉ kiểm trên truy vấn ngắn: với truy vấn/pattern dài `[a-z]*` lồng nhau có thể quay lui rất nhiều khi KHÔNG khớp
    short = [i for i, q in enumerate(queries) if len(q) <= 12 and len(pattern) <= 8]
    if short:
        sub = [queries[i] for i in short]
        assert camel_regex(sub, pattern) == sol_regex(sub, pattern) == [ref[i] for i in short], (sub[:3], pattern)
    if brute:
        for q, r in zip(queries, ref):
            if len(q) <= 14:
                assert match_brute(q, pattern) == r, (q, pattern)
    if expected is not None:
        assert ref == expected, (queries, pattern, ref, expected)
    return ref

# 12.1 ví dụ đề
for queries, pattern, expected in examples:
    check_all(queries, pattern, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    (["a"], "a", [True]),                              # khớp đúng
    (["A"], "A", [True]),
    (["a"], "A", [False]),                             # khác hoa thường
    (["A"], "a", [False]),
    (["Aa"], "A", [True]),                             # chèn chữ thường sau
    (["aA"], "A", [True]),                             # chèn chữ thường trước
    (["AA"], "A", [False]),                            # chữ hoa thừa
    (["AB"], "A", [False]),                            # chữ hoa thừa ở cuối
    (["a"], "aa", [False]),                            # query ngắn hơn pattern
    (["aaa"], "aa", [True]),                           # chèn 'a' (chữ thường) ở bất kỳ vị trí
    (["aXa"], "aa", [False]),                          # chữ hoa X chen giữa
    (["FooBar"], "FooBar", [True]),                    # trùng hoàn toàn
    (["FooBar"], "FBar", [True]),                      # chữ thường trong pattern
    (["FooBar"], "FB", [True]),
    (["FooBarTest"], "FB", [False]),                   # chữ hoa T thừa
    (["FrameBuffer"], "FoBa", [False]),                # chữ thường o, a của pattern không có đủ theo thứ tự
    (["aA", "Aa", "AA", "aa"], "A", [True, True, False, False]),   # 'aa' thiếu chữ hoa A
    (["abc"], "ac", [True]),                           # chỉ chữ thường
    (["abc"], "ca", [False]),                          # sai thứ tự
    (["xAyBz"], "AB", [True]),                         # chữ thường bao quanh
    (["xAyBzC"], "AB", [False]),                       # chữ hoa C ở cuối
]
for queries, pattern, expected in edge:
    check_all(queries, pattern, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 21 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm so với oracle theo định nghĩa)
def wrong_only_subsequence(queries, pattern):      # chỉ kiểm tra chuỗi con không liên tiếp, quên "ký tự thừa phải là chữ thường"
    return [is_subsequence(pattern, q) for q in queries]

def wrong_early_stop(queries, pattern):            # dừng sớm khi đã khớp hết pattern, bỏ qua chữ hoa còn lại
    res = []
    for q in queries:
        j, m, ok = 0, len(pattern), False
        for c in q:
            if j < m and c == pattern[j]:
                j += 1
                if j == m:
                    ok = True
                    break
            elif c.isupper():
                break
        res.append(ok)
    return res

def wrong_no_full_use(queries, pattern):           # quên kiểm tra đã dùng hết pattern
    res = []
    for q in queries:
        j, m, ok = 0, len(pattern), True
        for c in q:
            if j < m and c == pattern[j]:
                j += 1
            elif c.isupper():
                ok = False
                break
        res.append(ok)
    return res

def wrong_case_insensitive(queries, pattern):      # so sánh không phân biệt hoa/thường
    res = []
    for q in queries:
        j, m, ok = 0, len(pattern), True
        for c in q:
            if j < m and c.lower() == pattern[j].lower():
                j += 1
            elif c.isupper():
                ok = False
                break
        res.append(ok and j == m)
    return res

def wrong_only_uppercase(queries, pattern):        # chỉ so dãy chữ hoa của query và pattern
    up = [c for c in pattern if c.isupper()]
    return [[c for c in q if c.isupper()] == up for q in queries]

def find_counter(wrong, alphabet="abAB", max_q=5, max_p=3):
    for lq in range(1, max_q + 1):
        for qt in product(alphabet, repeat=lq):
            q = "".join(qt)
            for lp in range(1, max_p + 1):
                for pt in product(alphabet, repeat=lp):
                    p = "".join(pt)
                    if wrong([q], p) != [match_brute(q, p)]:
                        return q, p
    return None

counters = {}
for name, fn in (("chỉ kiểm tra chuỗi con (quên chữ hoa thừa)", wrong_only_subsequence), ("dừng sớm khi khớp hết pattern", wrong_early_stop),
                 ("quên kiểm tra dùng hết pattern", wrong_no_full_use), ("không phân biệt hoa/thường", wrong_case_insensitive),
                 ("chỉ so các chữ hoa", wrong_only_uppercase)):
    case = find_counter(fn)
    assert case is not None, name
    counters[name] = case
    q, p = case
    print(f"{name:44} query={q!r} pattern={p!r}: sai -> {fn([q], p)[0]}, đúng -> {match_brute(q, p)}")
    check_all([q], p)
print("12.3 phản ví dụ: OK")

chỉ kiểm tra chuỗi con (quên chữ hoa thừa)   query='aA' pattern='a': sai -> True, đúng -> False
dừng sớm khi khớp hết pattern                query='aA' pattern='a': sai -> True, đúng -> False
quên kiểm tra dùng hết pattern               query='a' pattern='b': sai -> True, đúng -> False
không phân biệt hoa/thường                   query='a' pattern='A': sai -> True, đúng -> False
chỉ so các chữ hoa                           query='a' pattern='b': sai -> True, đúng -> False
12.3 phản ví dụ: OK


In [21]:
# 12.4 ngẫu nhiên: query dài tới 100 so với DP (oracle độc lập với tham lam), nhiều bảng chữ cái
rng = random.Random(4023)
n_true = n_false = 0
for _ in range(1500):
    alphabet = rng.choice(["ab", "aAbB", "abcABC", "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"])
    pattern = "".join(rng.choice(alphabet) for _ in range(rng.randint(1, 8)))
    queries = []
    for _ in range(rng.randint(1, 10)):
        mode = rng.random()
        if mode < 0.5:                           # cài sẵn: chèn chữ thường vào pattern
            chars = list(pattern)
            for _ in range(rng.randint(0, 20)):
                chars.insert(rng.randint(0, len(chars)), rng.choice([c for c in alphabet if c.islower()] or ["a"]))
            queries.append("".join(chars))
        else:                                    # hoàn toàn ngẫu nhiên
            queries.append("".join(rng.choice(alphabet) for _ in range(rng.randint(1, 30))))
    ref = check_all(queries, pattern)
    n_true += sum(ref)
    n_false += len(ref) - sum(ref)
print(f"12.4 ngẫu nhiên: {n_true} truy vấn khớp, {n_false} không khớp: OK")
assert n_true > 100 and n_false > 100

# 12.5 cỡ lớn đúng giới hạn: 100 truy vấn độ dài 100, pattern độ dài tới 100
rng = random.Random(5023)
for _ in range(6):
    pattern = "".join(rng.choice("abAB") for _ in range(rng.randint(1, 100)))
    queries = []
    for _ in range(100):
        chars = list(pattern)
        while len(chars) < rng.randint(len(pattern), 100):
            chars.insert(rng.randint(0, len(chars)), rng.choice("ab"))
        queries.append("".join(chars[:100]))
    ref = check_all(queries, pattern, brute=False)
print("12.5 cỡ lớn (100 truy vấn x độ dài 100): OK")

# 12.6 tính chất: khớp chính nó; chèn thêm chữ thường vào truy vấn khớp vẫn khớp; chèn thêm chữ hoa thì hết khớp; khớp bắc cầu theo độ dài
rng = random.Random(6023)
for _ in range(500):
    pattern = "".join(rng.choice("abAB") for _ in range(rng.randint(1, 8)))
    assert sol.camelMatch([pattern], pattern) == [True]
    q = list(pattern)
    for _ in range(rng.randint(0, 6)):
        q.insert(rng.randint(0, len(q)), rng.choice("ab"))
    assert sol.camelMatch(["".join(q)], pattern) == [True]
    q2 = q[:]
    q2.insert(rng.randint(0, len(q2)), rng.choice("AB"))
    # thêm một chữ hoa vào chuỗi khớp: chỉ còn khớp nếu pattern có thêm chữ hoa đó (ta so với oracle định nghĩa)
    assert sol.camelMatch(["".join(q2)], pattern) == [match_dp("".join(q2), pattern)]
print("12.6 tính chất: OK")

12.4 ngẫu nhiên: 4884 truy vấn khớp, 3385 không khớp: OK
12.5 cỡ lớn (100 truy vấn x độ dài 100): OK
12.6 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm tra `pattern` là chuỗi con không liên tiếp

Quên điều kiện "ký tự thừa phải là **chữ thường**": `"FooBarTest"` chứa `"FB"` như chuỗi con nhưng có chữ hoa `T` thừa.

In [22]:
q, p = counters["chỉ kiểm tra chuỗi con (quên chữ hoa thừa)"]
print(f"query={q!r} pattern={p!r}: sai ->", wrong_only_subsequence([q], p)[0], "| đúng ->", Solution().camelMatch([q], p)[0])
assert wrong_only_subsequence([q], p) != Solution().camelMatch([q], p)
print("ví dụ 1:", wrong_only_subsequence(QUERIES, "FB"), "(sai) so với", Solution().camelMatch(QUERIES, "FB"))

query='aA' pattern='a': sai -> True | đúng -> False
ví dụ 1: [True, True, True, True, True] (sai) so với [True, False, True, True, False]


Vì sao sai: ví dụ 1 chỉ ra `"FooBarTest"` phải `false`. **Đúng:** thêm nhánh `elif c.isupper(): return False`.

### Lỗi 2 — Dừng sớm khi đã khớp hết `pattern`

Sau khi dùng hết `pattern` vẫn phải kiểm tra các ký tự còn lại của truy vấn: nếu còn chữ hoa thì không khớp.

In [23]:
q, p = counters["dừng sớm khi khớp hết pattern"]
print(f"query={q!r} pattern={p!r}: sai ->", wrong_early_stop([q], p)[0], "| đúng ->", Solution().camelMatch([q], p)[0])
assert wrong_early_stop([q], p) != Solution().camelMatch([q], p)

query='aA' pattern='a': sai -> True | đúng -> False


Vì sao sai: các ký tự sau phần khớp cũng là ký tự "thừa" nên phải là chữ thường. **Đúng:** duyệt hết truy vấn.

### Lỗi 3 — Quên kiểm tra đã dùng hết `pattern`

Duyệt hết truy vấn không gặp chữ hoa thừa nhưng `pattern` chưa khớp hết (truy vấn thiếu ký tự).

In [24]:
q, p = counters["quên kiểm tra dùng hết pattern"]
print(f"query={q!r} pattern={p!r}: sai ->", wrong_no_full_use([q], p)[0], "| đúng ->", Solution().camelMatch([q], p)[0])
assert wrong_no_full_use([q], p) != Solution().camelMatch([q], p)

query='a' pattern='b': sai -> True | đúng -> False


Vì sao sai: `pattern` phải là chuỗi con không liên tiếp **đầy đủ** của truy vấn. **Đúng:** `return j == len(pattern)`.

### Lỗi 4 — So sánh không phân biệt hoa/thường

`"a"` và `"A"` là các ký tự khác nhau: chữ thường chèn được, chữ hoa thì không.

In [25]:
q, p = counters["không phân biệt hoa/thường"]
print(f"query={q!r} pattern={p!r}: sai ->", wrong_case_insensitive([q], p)[0], "| đúng ->", Solution().camelMatch([q], p)[0])
assert wrong_case_insensitive([q], p) != Solution().camelMatch([q], p)

query='a' pattern='A': sai -> True | đúng -> False


Vì sao sai: `lower()` làm mất phân biệt `A` và `a` nên sai khi pattern có chữ hoa mà truy vấn chỉ có chữ thường tương ứng (hoặc ngược lại). **Đúng:** so sánh ký tự nguyên bản `c == pattern[j]`.

### Lỗi 5 — Chỉ so các chữ hoa

Chữ thường của `pattern` cũng phải xuất hiện đúng thứ tự trong truy vấn.

In [26]:
q, p = counters["chỉ so các chữ hoa"]
print(f"query={q!r} pattern={p!r}: sai ->", wrong_only_uppercase([q], p)[0], "| đúng ->", Solution().camelMatch([q], p)[0])
assert wrong_only_uppercase([q], p) != Solution().camelMatch([q], p)
print("ví dụ 2 (FoBa):", wrong_only_uppercase(QUERIES, "FoBa"), "(sai) so với", Solution().camelMatch(QUERIES, "FoBa"))

query='a' pattern='b': sai -> True | đúng -> False
ví dụ 2 (FoBa): [True, False, True, True, False] (sai) so với [True, False, True, False, False]


Vì sao sai: `"FrameBuffer"` có dãy chữ hoa `FB` nhưng không chứa `o`, `a` đúng thứ tự cho `"FoBa"`. **Đúng:** hai con trỏ trên toàn bộ ký tự.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "pattern là chuỗi con không liên tiếp của chuỗi, với ràng buộc về loại ký tự thừa" ⇒ **hai con trỏ** tham lam.
- Khớp CamelCase: dãy chữ hoa phải khớp từng chữ; chữ thường tự do **chen** giữa các chữ cần khớp.
- Tham lam đúng vì khớp sớm nhất chỉ để lại **nhiều chỗ** hơn cho phần sau (trao đổi).
- Luôn kiểm tra điều kiện kết thúc: đã dùng hết `pattern`.
- Cách khác: **regex** `[a-z]*P0[a-z]*P1...[a-z]*`, hoặc **DP** theo `(i, j)` để đối chiếu.
- Cùng họ: Is Subsequence (392), Wildcard Matching (44), Regular Expression Matching (10), Valid Word Abbreviation (408).
- Test nên có: chữ hoa thừa ở đầu/giữa/cuối, `pattern` dài hơn truy vấn, truy vấn trùng `pattern`, `pattern` toàn chữ thường, toàn chữ hoa.

## 15. Biến thể

1. **Cho phép tối đa `k` chữ hoa thừa:** thay `return False` bằng đếm số chữ hoa thừa `<= k`.
2. **Chèn bất kỳ ký tự nào** (không chỉ chữ thường): chỉ cần `pattern` là chuỗi con không liên tiếp (bài 392).
3. **Cho biết một cách ghép** (chỉ số khớp): bài tập 5.
4. Bài liên quan: 392, 44, 10.

Biến thể 1 (cho phép tối đa `k` chữ hoa thừa) kiểm bằng DP có ngân sách:

In [27]:
def match_budget(q: str, pattern: str, k: int) -> bool:
    j, m, used = 0, len(pattern), 0
    for c in q:
        if j < m and c == pattern[j]:
            j += 1
        elif c.isupper():
            used += 1
            if used > k:
                return False
    return j == m

def match_budget_dp(q: str, pattern: str, k: int) -> bool:
    n, m = len(q), len(pattern)

    @lru_cache(None)
    def f(i, j, used):
        if used > k:
            return False
        if i == n:
            return j == m
        if j < m and q[i] == pattern[j] and f(i + 1, j + 1, used):
            return True
        return f(i + 1, j, used + (1 if q[i].isupper() else 0))

    return f(0, 0, 0)

rng = random.Random(15)
for _ in range(1500):
    p = "".join(rng.choice("abAB") for _ in range(rng.randint(1, 4)))
    q = "".join(rng.choice("abAB") for _ in range(rng.randint(1, 9)))
    k = rng.randint(0, 3)
    assert match_budget(q, p, k) == match_budget_dp(q, p, k), (q, p, k)
    if k == 0:
        assert match_budget(q, p, 0) == match_pointers(q, p)
print("biến thể cho phép k chữ hoa thừa: hai con trỏ khớp DP có ngân sách; k = 0 trùng bài gốc")

biến thể cho phép k chữ hoa thừa: hai con trỏ khớp DP có ngân sách; k = 0 trùng bài gốc


## 16. So sánh các cách

| Cách | Ý tưởng | Time (mỗi truy vấn) | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi tập chèn | Theo định nghĩa | `O(C(\|q\|, \|q\| - \|p\|) * \|q\|)` | `O(\|q\|)` | Oracle, truy vấn ngắn |
| 2. DP `(i, j)` | Cả hai lựa chọn | `O(\|q\| * \|p\|)` | `O(\|q\| * \|p\|)` | Đối chiếu độc lập |
| 3. Hai con trỏ | Tham lam | `O(\|q\|)` | `O(1)` | **Cách nộp chính** |
| 4. Regex | `[a-z]*P0[a-z]*...` | `O(\|q\| * \|p\|)` tồi nhất | `O(\|p\|)` | Viết ngắn |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print("F".isupper(), "f".isupper(), "ab".islower(), "aB".islower())
print(is_subsequence("ac", "abc"), is_subsequence("ca", "abc"))
print(Solution().camelMatch(["Ab", "AB", "aA"], "A"))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # bốn bool
my_guess_2 = None      # hai bool
my_guess_3 = None      # danh sách ba bool

### Bài 2 — Tính tay (dễ-vừa)

`pattern = "FoB"`, `queries = ["FooBar", "FoBr", "FBo"]`. Ghi mảng kết quả.

In [29]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hai con trỏ.

In [30]:
def exercise_3(queries: List[str], pattern: str) -> List[bool]:
    def match(q: str) -> bool:
        j, m = 0, len(pattern)
        for c in q:
            # TODO 1: nếu j < m và c == pattern[j] thì j += 1
            # TODO 2: ngược lại nếu c là chữ hoa thì trả về False
            pass
        # TODO 3: trả về j == m
        pass

    return [match(q) for q in queries]


# for queries, pattern, expected in examples:
#     assert exercise_3(queries, pattern) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, tự viết `exercise_4(queries, pattern)` bằng **biểu thức chính quy** (`[a-z]*` xen các ký tự của `pattern`).

In [31]:
def exercise_4(queries: List[str], pattern: str) -> List[bool]:
    # TODO
    pass


# for queries, pattern, expected in examples:
#     assert exercise_4(queries, pattern) == expected

### Bài 5 — Mini challenge: tìm một cách ghép (khó)

Viết `exercise_5(query, pattern)` trả về **danh sách các chỉ số** `[i_0, i_1, ..., i_{m-1}]` trong `query` được ghép với các ký tự của `pattern` (tăng dần, `query[i_k] == pattern[k]`, mọi ký tự còn lại là chữ thường), hoặc `None` nếu `query` không khớp. Có thể có nhiều cách ghép hợp lệ; ta kiểm tra bằng **hàm kiểm tra** (không so sánh với một đáp án cố định).

```python
exercise_5("FooBar", "FB")        -> [0, 3]
exercise_5("FooBarTest", "FB")    -> None
exercise_5("FooBar", "FoBa")      -> [0, 1, 3, 4] hoặc [0, 2, 3, 4]
```
Gợi ý: hai con trỏ, ghi lại chỉ số mỗi lần khớp.

In [32]:
def exercise_5(query: str, pattern: str) -> Optional[List[int]]:
    # TODO
    pass


def check_alignment(query: str, pattern: str, idx: Optional[List[int]]) -> bool:
    """Hàm kiểm tra: idx là cách ghép hợp lệ, hoặc None khi (và chỉ khi) không khớp."""
    if idx is None:
        return not match_pointers(query, pattern)
    return (len(idx) == len(pattern) and all(a < b for a, b in zip(idx, idx[1:])) and all(0 <= i < len(query) for i in idx)
            and all(query[i] == c for i, c in zip(idx, pattern))
            and all(query[i].islower() for i in range(len(query)) if i not in set(idx)))


# for q, p in [("FooBar", "FB"), ("FooBarTest", "FB"), ("FooBar", "FoBa"), ("a", "A")]:
#     assert check_alignment(q, p, exercise_5(q, p))

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `isupper()` đúng khi mọi chữ cái là chữ hoa (và có ít nhất một chữ cái); `islower()` tương tự cho chữ thường.
- Mức 2: `"aB".islower()` là `False` (có chữ hoa); `is_subsequence("ca", "abc")` sai (sai thứ tự).
- Mức 3: dòng 1: `True False True False`; dòng 2: `True False`; dòng 3: `[True, False, True]` (`"aA"`: `a` thừa thường, `A` khớp).

**Bài 2**
- Mức 1: `"FooBar"`: `F`, `o` khớp; `o` thừa thường; `B` khớp; `a`, `r` thừa thường.
- Mức 2: `"FoBr"`: `F`, `o`, `B` khớp; `r` thừa. `"FBo"`: `F` khớp; `B` không khớp `o` và là chữ hoa thừa.
- Mức 3: `[True, True, False]`.

**Bài 3**
- Mức 1: `if j < m and c == pattern[j]: j += 1`.
- Mức 2: `elif c.isupper(): return False`.
- Mức 3: `return j == m`.

**Bài 4**
- Mức 1: `rx = "[a-z]*" + "[a-z]*".join(pattern) + "[a-z]*"`.
- Mức 2: `re.compile(rx)`; dùng `fullmatch` cho từng truy vấn.
- Mức 3: `[rx.fullmatch(q) is not None for q in queries]`.

**Bài 5**
- Mức 1: `idx = []`, `j = 0`.
- Mức 2: duyệt `i, c`: nếu `j < m and c == pattern[j]`: `idx.append(i); j += 1`; elif `c.isupper()`: trả `None`.
- Mức 3: cuối cùng `return idx if j == m else None`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
print("F".isupper(), "f".isupper(), "ab".islower(), "aB".islower())
print(is_subsequence("ac", "abc"), is_subsequence("ca", "abc"))
print(Solution().camelMatch(["Ab", "AB", "aA"], "A"))

assert ("F".isupper(), "f".isupper(), "ab".islower(), "aB".islower()) == (True, False, True, False)
assert (is_subsequence("ac", "abc"), is_subsequence("ca", "abc")) == (True, False)
assert Solution().camelMatch(["Ab", "AB", "aA"], "A") == [True, False, True]

True False True False
True False
[True, False, True]


### Đáp án Bài 2

`"FooBar"`: khớp (`F`, `o`, `B`; `o`, `a`, `r` thừa thường). `"FoBr"`: khớp (`r` thừa thường). `"FBo"`: sau `F`, ký tự `B` không bằng `o` và là chữ hoa thừa: không khớp.

In [34]:
hand_answer_solution = [True, True, False]
assert Solution().camelMatch(["FooBar", "FoBr", "FBo"], "FoB") == hand_answer_solution
assert camel_brute(["FooBar", "FoBr", "FBo"], "FoB") == hand_answer_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_solution(queries: List[str], pattern: str) -> List[bool]:
    def match(q: str) -> bool:
        j, m = 0, len(pattern)
        for c in q:
            if j < m and c == pattern[j]:
                j += 1
            elif c.isupper():
                return False
        return j == m

    return [match(q) for q in queries]

for queries, pattern, expected in examples:
    assert exercise_3_solution(queries, pattern) == expected
rng = random.Random(3)
for _ in range(300):
    p = "".join(rng.choice("abAB") for _ in range(rng.randint(1, 4)))
    qs = ["".join(rng.choice("abAB") for _ in range(rng.randint(1, 8))) for _ in range(5)]
    assert exercise_3_solution(qs, p) == camel_brute(qs, p)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_solution(queries: List[str], pattern: str) -> List[bool]:
    rx = re.compile("[a-z]*" + "[a-z]*".join(re.escape(c) for c in pattern) + "[a-z]*")
    return [rx.fullmatch(q) is not None for q in queries]

for queries, pattern, expected in examples:
    assert exercise_4_solution(queries, pattern) == expected
rng = random.Random(4)
for _ in range(300):
    p = "".join(rng.choice("abAB") for _ in range(rng.randint(1, 4)))
    qs = ["".join(rng.choice("abAB") for _ in range(rng.randint(1, 8))) for _ in range(5)]
    assert exercise_4_solution(qs, p) == camel_brute(qs, p)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_solution(query: str, pattern: str) -> Optional[List[int]]:
    idx, j, m = [], 0, len(pattern)
    for i, c in enumerate(query):
        if j < m and c == pattern[j]:
            idx.append(i)
            j += 1
        elif c.isupper():
            return None
    return idx if j == m else None

assert exercise_5_solution("FooBar", "FB") == [0, 3]
assert exercise_5_solution("FooBarTest", "FB") is None
assert check_alignment("FooBar", "FoBa", exercise_5_solution("FooBar", "FoBa"))
rng = random.Random(5)
for _ in range(500):
    p = "".join(rng.choice("abAB") for _ in range(rng.randint(1, 4)))
    q = "".join(rng.choice("abAB") for _ in range(rng.randint(1, 9)))
    assert check_alignment(q, p, exercise_5_solution(q, p)), (q, p)
    assert (exercise_5_solution(q, p) is not None) == match_brute(q, p)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (hai con trỏ)

```python
class Solution:
    def camelMatch(self, queries: List[str], pattern: str) -> List[bool]:
        def match(q: str) -> bool:
            j, m = 0, len(pattern)
            for c in q:
                if j < m and c == pattern[j]:
                    j += 1                      # khớp ký tự pattern sớm nhất có thể
                elif c.isupper():
                    return False                # chữ hoa thừa: không thể là ký tự chèn
            return j == m                       # phải dùng hết pattern

        return [match(q) for q in queries]
```

Công thức nhớ nhanh: khớp ⇔ `pattern` là chuỗi con không liên tiếp **và** mọi ký tự thừa là chữ thường; hai con trỏ: khớp khi bằng, loại khi chữ hoa thừa, cuối cùng `j == len(pattern)`.

### Checklist trước khi submit

- [ ] Có nhánh "chữ hoa không khớp thì trả `False`" (ký tự thừa phải là chữ thường)?
- [ ] Duyệt **hết** truy vấn (không dừng sớm khi đã khớp hết `pattern`)?
- [ ] Cuối cùng kiểm tra `j == len(pattern)`?
- [ ] So sánh ký tự **nguyên bản** (phân biệt hoa/thường)?
- [ ] Chữ thường của `pattern` cũng được khớp đúng thứ tự (không chỉ chữ hoa)?
- [ ] Đã thử: chữ hoa thừa ở đầu/giữa/cuối, `pattern` dài hơn truy vấn, truy vấn trùng `pattern`, `|q| = 100`?